# IBL - Brain Wide Map

References:
- [IBL Brain Wide Map](https://dandiarchive.org/dandiset/000409/draft)
- [Explore IBL sessions](https://viz.internationalbrainlab.org/app)

In [ ]:
!python -m venv .venv && \
source .venv/bin/activate && \
pip install -r requirements.txt && \
python -m ipykernel install --user --name=.venv --display-name "Python (.venv)"

## Create a Pipeline

Brainsets are created using [Pipelines](https://brainsets.readthedocs.io/en/latest/concepts/create_pipeline.html). You can check the implementation details for the IBL Pipeline in `pipeline.py`.

This Pipeline accepts the following arguments:
- **interval_ref_time**: the intra-trial event to be used as a reference starting time for each interval chunk
- **interval_max_duration**: the maximum duration for each interval chunk

Intervals are chunks of time along a session which mark the periods of time to be used by training, validation and test data. The Pipeline code will automatically create these splits.

For this experimental dataset, the onset time of visual stimulus seems to be a good reference point, with the motor areas activity sustained for around 1 second, so these can be sensible choices when creating the Brainset Intervals from NWB trials.

<figure>
  <img src="images/raster_plots.png" width="800"/>
  <figcaption>Figure 1: Stimulus onset is the best reference event for motor related neural activity.</figcaption>
</figure>

In [11]:
from pathlib import Path
from argparse import Namespace
from pipeline import Pipeline

# Define directories
raw_dir = Path("./ibl_nwb").resolve()
processed_dir = Path("./ibl_processed").resolve()

# Create args namespace (simulating command-line arguments)
args = Namespace(
    interval_ref_time="gabor_stimulus_onset_time",
    interval_max_duration=1.0,
    redownload=False,
    reprocess=True,
)

# Instantiate the pipeline
pipeline = Pipeline(
    raw_dir=raw_dir,
    processed_dir=processed_dir,
    args=args,
)

# Get the manifest
manifest = Pipeline.get_manifest(raw_dir, args)
manifest

,filename
session_id,
sub-NYU-21_ses-8c33abef-3d3e-4d42-9f27-445e9def08f9,sub-NYU-21_ses-8c33abef-3d3e-4d42-9f27-445e9de...
sub-CSHL059_ses-d2f5a130-b981-4546-8858-c94ae1da75ff,sub-CSHL059_ses-d2f5a130-b981-4546-8858-c94ae1...
sub-UCLA035_ses-6f36868f-5cc1-450c-82fa-6b9829ce0cfe,sub-UCLA035_ses-6f36868f-5cc1-450c-82fa-6b9829...
sub-DY-011_ses-7bee9f09-a238-42cf-b499-f51f765c6ded,sub-DY-011_ses-7bee9f09-a238-42cf-b499-f51f765...
sub-DY-014_ses-bd456d8f-d36e-434a-8051-ff3997253802,sub-DY-014_ses-bd456d8f-d36e-434a-8051-ff39972...
sub-NR-0027_ses-ae8787b1-4229-4d56-b0c2-566b61a25b77,sub-NR-0027_ses-ae8787b1-4229-4d56-b0c2-566b61...
sub-SWC-038_ses-03063955-2523-47bd-ae57-f7489dd40f15,sub-SWC-038_ses-03063955-2523-47bd-ae57-f7489d...
sub-ZFM-01936_ses-4aa1d525-5c7d-4c50-a147-ec53a9014812,sub-ZFM-01936_ses-4aa1d525-5c7d-4c50-a147-ec53...
sub-CSH-ZAD-026_ses-81a78eac-9d36-4f90-a73a-7eb3ad7f770b,sub-CSH-ZAD-026_ses-81a78eac-9d36-4f90-a73a-7e...


## Process sessions

Now we are ready to process all sessions. This will automatically:
- download the nwb files from DANDI, to the `ibl_nwb` folder
- extract the relevant content from nwb files
- save the results as a Brainset, to the `ibl_processed` folder

In [ ]:
# Process sessions
for manifest_item in manifest.itertuples():
    print(f"Processing session: {manifest_item.Index}")
    pipeline._run_item(manifest_item)

## Investigate Brainset data

We can now take a look at what type of data is present in the processed Brainset:

In [6]:
import h5py
from temporaldata import Data


f = h5py.File("ibl_processed/sub-CSHL059_ses-d2f5a130-b981-4546-8858-c94ae1da75ff_desc-processed_behavior+ecephys.h5", "r")
session_data = Data.from_hdf5(f, lazy=True)
train_data = session_data.select_by_interval(session_data.train_domain)
print(train_data.keys())

['brainset', 'device', 'pose_estimation_left_camera', 'pose_estimation_right_camera', 'session', 'spikes', 'subject', 'test_domain', 'train_domain', 'trials', 'units', 'valid_domain', 'wheel_acceleration', 'wheel_movement_intervals', 'wheel_position', 'wheel_velocity']


In [7]:
train_data.brainset

Data(
brainsets_version='0.2.1.dev2+g951acd758',
derived_version='1.0.0',
description='International Brain Laboratory Brain Wide Map dataset containing Neuropixels recordings across multiple brain regions in mice performing a visual discrimination task.',
id='ibl_brain_wide_map_2026',
origin_version='draft',
source='https://dandiarchive.org/dandiset/000409/draft',
temporaldata_version='0.1.1',
_absolute_start=0.0,
)

In [10]:
train_data.trials

LazyInterval(
  auditory_cue_time=<HDF5 dataset "auditory_cue_time": shape (525,), type "<f8">,
  block_index=<HDF5 dataset "block_index": shape (525,), type "<i8">,
  block_type=<HDF5 dataset "block_type": shape (525,), type "|O">,
  choice_registration_time=<HDF5 dataset "choice_registration_time": shape (525,), type "<f8">,
  end=[368],
  feedback_time=<HDF5 dataset "feedback_time": shape (525,), type "<f8">,
  gabor_stimulus_contrast=<HDF5 dataset "gabor_stimulus_contrast": shape (525,), type "<f8">,
  gabor_stimulus_offset_time=<HDF5 dataset "gabor_stimulus_offset_time": shape (525,), type "<f8">,
  gabor_stimulus_onset_time=<HDF5 dataset "gabor_stimulus_onset_time": shape (525,), type "<f8">,
  gabor_stimulus_side=<HDF5 dataset "gabor_stimulus_side": shape (525,), type "|O">,
  is_mouse_rewarded=<HDF5 dataset "is_mouse_rewarded": shape (525,), type "|b1">,
  mouse_wheel_choice=<HDF5 dataset "mouse_wheel_choice": shape (525,), type "|O">,
  probability_left=<HDF5 dataset "probabil